# Score Shaping & Pareto Optimality — interactive figures
**NEWCAS 2026 · telescopic OTA (IHP-130nm) · Differential Evolution**

Four interactive figures, driven by the *real* SpiceXplorer runs in `./data/`, that
show how the **score-shaping** strategy turns a set of competing specs into a single
landscape the optimizer can descend — and why the **sigmoid** shape enforces a
balanced trade-off where the **linear** baseline does not.

---

## Problem formulation

We size a design vector $\mathbf{x}\in\mathcal{X}\subset\mathbb{R}^{14}$ (transistor
widths, lengths, bias voltages). SPICE maps it to a performance vector
$\mathbf{m}(\mathbf{x})=(m_1,\dots,m_K)$, here $K=6$ (Table I). Each metric $i$ has a
target $T_i$ and a type $\tau_i\in\{\textsf{EXCEED},\textsf{MINIMIZE},\textsf{EXACT}\}$.

The problem is cast as a **single scalar objective** $F(\mathbf{x})$ — not a
multi-objective Pareto search — that *encodes* constraint satisfaction inside one
continuous score. While any spec is violated, $F$ is dominated by penalties (the
constraint-satisfaction phase); the moment every penalty reaches zero the objective
switches branches and keeps **accumulating positive rewards** for exceeding spec (the
figure-of-merit phase). Feasibility is therefore a *hand-off point*, not a stopping
point — the optimizer keeps improving past it:

$$\max_{\mathbf{x}\in\mathcal{X}}\ F(\mathbf{x}),
\qquad
\underbrace{\sum_i p_i(\mathbf{x})=0}_{\text{feasible } \mathcal{F}}
\ \Longrightarrow\
F(\mathbf{x})>0\ \text{ and rising.}$$

**Raw penalty** per metric (Sec. III-A) measures the signed shortfall:

$$
p_i(\mathbf{x})=
\begin{cases}
\max\!\big(0,\;T_i-m_i\big) & \tau_i=\textsf{EXCEED}\quad(\text{gain, bandwidth})\\[2pt]
\max\!\big(0,\;m_i-T_i\big) & \tau_i=\textsf{MINIMIZE}\quad(\text{power, noise})\\[2pt]
\bigl|\,m_i-T_i\,\bigr| & \tau_i=\textsf{EXACT}\quad(\text{phase margin}=60^\circ)
\end{cases}
$$

**Normalization** removes the scale mismatch between metrics
($10^{9}\,\text{Hz}$ vs. $10^{-6}\,\text{A}$):

$$
\hat P_i=\frac{p_i}{S_i},\qquad S_i=\lvert T_i\rvert
\qquad\text{(Eq. 1, linear baseline).}
$$

**Sigmoid shaping** (Eq. 2, proposed) then squashes the relative error into $[0,1)$:

$$
P_i=\frac{2}{1+e^{-\alpha\hat P_i}}-1
   \;=\;\tanh\!\Big(\tfrac{\alpha}{2}\,\hat P_i\Big)\in[0,1),
$$

using the identity $\tfrac{2}{1+e^{-u}}-1=\tanh(u/2)$.

**Aggregation** — the dual-mode single objective (Eq. 3): penalize while infeasible,
then reward beyond feasibility:

$$
F(\mathbf{x})=
\begin{cases}
\;+\displaystyle\sum_i w_i\,R_i(m_i), & \displaystyle\sum_i P_i(m_i)\approx 0\quad(\textit{feasible — accumulate reward})\\[10pt]
\;-\displaystyle\sum_i w_i\,P_i(m_i), & \text{otherwise}\quad(\textit{infeasible — penalize}).
\end{cases}
$$

The optimizer **maximizes** the single score $F$. The two branches meet at the
feasibility boundary $\sum_i P_i=0$: below it $F<0$ (penalty-driven), at it $F=0$, and
above it $F>0$ and climbing as rewards stack up. So the zero-crossing is the switch
from *"become feasible"* to *"now get better"* — in Fig. 3 the sigmoid trace crosses
zero and keeps rising, while the linear trace never reaches it. Everything below studies
how the choice of $P_i$ — linear $\hat P_i$ vs. sigmoid
$\tanh(\tfrac{\alpha}{2}\hat P_i)$ — reshapes this objective.

In [ ]:
import importlib
import score_shaping as ss
import build_plots as bp
importlib.reload(ss); importlib.reload(bp)   # re-run after editing the .py files

import plotly.io as pio
pio.renderers.default = "notebook"            # inline, self-contained in the notebook
print("metrics:", ", ".join(ss.METRICS))

## 1 · The squashing function $P_i(\hat P_i)$

The two normalizations differ only in how they map the relative error $\hat P_i$ to a
penalty:

$$
\textbf{Linear: } P_i=\hat P_i\in[0,\infty)
\qquad\qquad
\textbf{Sigmoid: } P_i=\tanh\!\Big(\tfrac{\alpha}{2}\,\hat P_i\Big)\in[0,1).
$$

Differentiating gives the **gradient the optimizer actually feels** per unit of
relative error:

$$
\frac{dP_i}{d\hat P_i}\Big|_{\text{lin}}=1
\qquad\qquad
\frac{dP_i}{d\hat P_i}\Big|_{\text{sig}}
=\frac{\alpha}{2}\,\operatorname{sech}^2\!\Big(\tfrac{\alpha}{2}\hat P_i\Big)
=\frac{\alpha}{2}\bigl(1-P_i^2\bigr).
$$

| property | linear | sigmoid |
|---|---|---|
| range | $[0,\infty)$ — unbounded | $[0,1)$ — **bounded** |
| slope at boundary $\hat P_i{=}0$ | $1$ | $\alpha/2$ |
| slope as $\hat P_i\to\infty$ | $1$ (constant) | $\to 0$ (**saturates**) |

Because the sigmoid slope $\tfrac{\alpha}{2}(1-P_i^2)\to0$, a metric *far* from
compliance contributes a near-constant $P_i\approx1$ and **cannot dominate** the
aggregate — drag $\alpha$ to set how fast that saturation kicks in.

In [ ]:
bp.fig_squash()

## 2 · The masking effect

Consider two metrics: a **near-feasible** one held at $\hat P_n=0.30$ and an
**outlier** at $\hat P_o$ that drifts away from target. The fraction of the total
penalty the near-feasible metric retains — i.e. how much *attention* the optimizer
pays it — is

$$
\sigma_n=\frac{P_n}{P_n+P_o}.
$$

Taking $\hat P_o\to\infty$ exposes the difference:

$$
\sigma_n^{\,\text{lin}}=\frac{\hat P_n}{\hat P_n+\hat P_o}\;\xrightarrow[\hat P_o\to\infty]{}\;0
\qquad\Longleftarrow\ \text{the violation is \textbf{masked}},
$$

$$
\sigma_n^{\,\text{sig}}=\frac{\tanh(\tfrac{\alpha}{2}\hat P_n)}{\tanh(\tfrac{\alpha}{2}\hat P_n)+\tanh(\tfrac{\alpha}{2}\hat P_o)}
\;\xrightarrow[\hat P_o\to\infty]{}\;
\frac{\tanh(\tfrac{\alpha}{2}\hat P_n)}{\tanh(\tfrac{\alpha}{2}\hat P_n)+1}\;>\;0.
$$

The sigmoid guarantees a **nonzero attention floor** on the violated metric (with
$\hat P_n{=}0.30,\ \alpha{=}1$ that floor is $\tanh(0.15)/(\tanh(0.15)+1)\approx13\%$),
whereas the linear share decays to $0$. This is exactly why the linear run never
repaired its current budget. Drag $\alpha$ to raise the floor.

In [ ]:
bp.fig_masking()

## 3 · The trade-off outcome

Define a dimensionless **spec margin** $\mu_i$ (achieved $\div$ target, oriented so
larger is always better):

$$
\mu_i=
\begin{cases}
m_i/T_i & \textsf{EXCEED}\\
T_i/m_i & \textsf{MINIMIZE}\\
2-\lvert m_i-T_i\rvert/\delta_i & \textsf{EXACT}\ (\text{tol. }\delta_i)
\end{cases}
\qquad\Longrightarrow\qquad
\text{meets spec}\iff \mu_i\ge 1 .
$$

A design is feasible iff $\min_i \mu_i \ge 1$. From Table III:

$$
\text{linear: }\ \min_i \mu_i=\mu_{I_{DD}}=\frac{25}{43.7}=0.57<1\ \ (\textbf{infeasible}),
\qquad
\text{sigmoid: }\ \min_i \mu_i\ge 1\ \ (\textbf{feasible}).
$$

**Left:** the margins per metric — the linear best wins on noise and settling but its
$I_{DD}$ bar drops below the $\mu=1$ line. **Right:** the same designs in
$(f_{UGB},\,I_{DD})$ space; the linear cloud and star sit *above* the $25\ \mu$A line,
the sigmoid cloud lands in the feasible quadrant.

In [ ]:
bp.fig_outcome()

## 4 · Equi-score landscape — how the *shape* steers the optimizer

Restrict the cost to two metrics $m_x,m_y$ and write the **two-metric field**

$$
F_2(m_x,m_y)=P_x(m_x)+P_y(m_y),
\qquad
\text{equi-score set }\ \mathcal{C}_c=\{(m_x,m_y):F_2=c\}.
$$

The iso-score curves $\mathcal{C}_c$ are level sets of $F_2$, so their shape is governed
by the gradient

$$
\nabla F_2=\Big(\tfrac{\partial P_x}{\partial m_x},\,\tfrac{\partial P_y}{\partial m_y}\Big),
\qquad
\frac{\partial P_i}{\partial m_i}
=\underbrace{\frac{dP_i}{d\hat P_i}}_{\text{shape}}\;
 \underbrace{\frac{d\hat P_i}{dm_i}}_{=\,\mp 1/S_i}.
$$

**Linear** $\big(\tfrac{dP_i}{d\hat P_i}=1\big)$: the gradient is piecewise
**constant**, so $\mathcal{C}_c$ are **straight, parallel lines**. The axis with the
larger $\lvert\partial P/\partial m\rvert$ over its span dominates and flattens the
other — the masking of Section 2, now in 2-D.

**Sigmoid** $\big(\tfrac{dP_i}{d\hat P_i}=\tfrac{\alpha}{2}(1-P_i^2)\big)$: the gradient
**vanishes on a plateau** far from target ($P_i\to1$) and is **steepest near the
boundary** ($P_i\to0$). The level sets close into **concentric contours** — a basin
centered on the feasible corner: a *soft feasibility barrier*.

Toggle **Linear/Sigmoid**, drag $\alpha$, and read the labelled iso-score lines.
Real visited points and the two best designs are overlaid; dashed white lines mark the
targets (EXACT metrics show their $\pm\delta$ band as dotted lines).

In [ ]:
bp.fig_contours("ugf", "i(idd_total)")

### Pick any two metrics

The landscape above is defined for **any** pair $m_x,m_y$ from `ss.METRICS`. Use the
dropdowns to choose the two axes; the contour, target guides, and overlaid runs all
recompute. (Axes auto-select log scaling for strictly-positive metrics — UGF, noise,
current — and linear otherwise.)

> Try **Input Noise × Supply Current** (the noise/power trade-off the paper calls out),
> or **DC Gain × Phase Margin** to see an `EXACT` constraint band.

In [ ]:
from ipywidgets import interact, Dropdown

_opts = {f"{ss.METRICS[k]['label']}  [{ss.METRICS[k]['unit']}]": k for k in ss.METRICS}

@interact(x_metric=Dropdown(options=_opts, value="ugf", description="X axis"),
          y_metric=Dropdown(options=_opts, value="i(idd_total)", description="Y axis"))
def show_contours(x_metric, y_metric):
    if x_metric == y_metric:
        print("Choose two different metrics for the two axes.")
        return
    bp.fig_contours(x_metric, y_metric).show()

## Export self-contained HTML for the talk
Writes `./plots/{1..4}_*.html` (each works offline — no internet/kernel needed) plus a
`plots/index.html` launcher and PNG thumbnails. The exported contour uses the default
UGF × current pair; export another pair with
`bp.fig_contours("v(inoise_total)", "i(idd_total)").write_html("plots/custom.html", include_plotlyjs=True)`.

In [ ]:
bp.main()
print("\nOpen plots/index.html in a browser to present.")